# The Great American Coffee Taste Test : Predicting Daily Coffee Consumption
## STAT 301 — Group Final Report

**Group 9: Best Fits**
**Group Members:** Tiffany Nguyen, Makafui Amouzouvi, Adam Cook, Nolan Bishop
**Dataset:** The Great American Coffee Taste Test (TidyTuesday, 2024-05-14)

April 16, 2026

In [1]:
# Load required libraries
library(tidyverse)
library(knitr)
library(broom)
library(glmnet)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glmnet 4.1-8



---
# 1. Introduction

### Background

In October 2023, world-champion barista James Hoffmann and coffee company Cometeer hosted "The Great American Coffee Taste Test" as a YouTube livestream. Approximately 4,000 participants across the United States received four flash-frozen coffee samples and tasted them blind while following along with the stream. After tasting, participants completed a detailed online survey covering their demographics, coffee habits, and ratings for each of the four coffees. The data was subsequently made public by data blogger Robert McKeon Aloe and later published through the TidyTuesday project (week of May 14, 2024).

**Important caveat:** Because participation was voluntary and the audience consisted primarily of James Hoffmann's fanbase, the sample is not representative of the general American population.

**Citation:**  
McKeon Aloe, R. (2023). *Great American Coffee Taste Test Breakdown*. Medium. https://rmckeon.medium.com/great-american-coffee-taste-test-breakdown-7f3fdcc3c41d  
TidyTuesday (2024-05-14). https://github.com/rfordatascience/tidytuesday/blob/main/data/2024/2024-05-14/readme.md

**Research Question**

> ***Can we use demographic characteristics and coffee-drinking habits (age, self-reported coffee expertise, preferred roast level, preferred strength etc...) to predict the number of cups of coffee a person drinks per day?***

The question remains focused on **prediction**: whether a set of demographic and habit variables can meaningfully predict daily coffee consumption for new individuals. Because the data come from a voluntary survey (observational study), we cannot make causal claims. The question is framed strictly in terms of *prediction*. The primary goal is to build a model that can accurately estimate the daily coffee consumption count for new individuals. 

---
## 2. Methods and Results

## 2a. Data

In [3]:
# Load data directly from GitHub (reproducible — no local files)
coffee_survey <- readr::read_csv(
  'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2024/2024-05-14/coffee_survey.csv',
  show_col_types = FALSE
)

cat("Number of observations:", nrow(coffee_survey), "\n")
cat("Number of variables:", ncol(coffee_survey), "\n")


Number of observations: 4042 
Number of variables: 57 


In [4]:
# Create a readable table of variable names and types
var_table <- tibble(
  Variable = names(coffee_survey),
  Type = sapply(coffee_survey, class)
)

kable(var_table, caption = "Table 1: Variable names and data types in the coffee_survey dataset")




Table: Table 1: Variable names and data types in the coffee_survey dataset

|Variable                     |Type      |
|:----------------------------|:---------|
|submission_id                |character |
|age                          |character |
|cups                         |character |
|where_drink                  |character |
|brew                         |character |
|brew_other                   |character |
|purchase                     |character |
|purchase_other               |character |
|favorite                     |character |
|favorite_specify             |character |
|additions                    |character |
|additions_other              |character |
|dairy                        |character |
|sweetener                    |character |
|style                        |character |
|strength                     |character |
|roast_level                  |character |
|caffeine                     |character |
|expertise                    |numeric   |
|coffee_a_bitternes

The dataset contains **4,042 observations** and **57 variables**, covering participant demographics, coffee consumption habits, and blind taste-test ratings for four coffees.

**Response Variable:** The response variable is cups of coffee consumed per day (cups). 

**Data Collection**

The data were collected via a voluntary online survey linked to a YouTube livestream event on October 21, 2023. Participants self-selected into the study by purchasing a Cometeer coffee kit in advance. Because the sample is self-selected and drawn from the audience of a specific content creator, we assume the data represent a **convenience sample** and may not generalize to all American coffee drinkers.

**Variable types:** Most demographic and preference variables are stored as character strings (e.g., `age`, `roast_level`, `strength`). A small number are numeric (e.g., `expertise`, taste-test ratings). The response variable `cups` records self-reported daily consumption as an ordered string category.

**Variable exclusions:** The following variables were excluded from the candidate predictor set before modelling, with justification:
